# Audio Augmentation Testing

This notebook tests the audio augmentation methods from `data_augmentation.py` on the test audio files.

In [5]:
import os
import sys
import librosa
import numpy as np
import soundfile as sf
from pathlib import Path
import auxiliary as a
import utils as u

# Add the current directory to the path so we can import data_augmentation
sys.path.append('.')

# Import our augmentation functions
from data_augmentation import randomly_eq, apply_compression, add_noise

print("Libraries imported successfully!")

c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


In [2]:
# Create output folder
output_folder = 'augmentations_test_outputs'
os.makedirs(output_folder, exist_ok=True)

# List test audio files
test_audio_folder = 'test_audio'
audio_files = [f for f in os.listdir(test_audio_folder) if f.endswith('.mp3')]
print(f"Found {len(audio_files)} audio files in {test_audio_folder}:")
for audio_file in audio_files[:5]:  # Show first 5
    print(f"  - {audio_file}")
if len(audio_files) > 5:
    print(f"  ... and {len(audio_files) - 5} more")

Found 11 audio files in test_audio:
  - TRBFMXE149E3E18222.mp3
  - TRBKNSX149E2E1F89F.mp3
  - TRFVOOM127FA301CFB.mp3
  - TRKINMI127FA68F716.mp3
  - TRMLNPR149E35EF902.mp3
  ... and 6 more


In [3]:
# Define the augmentations to test
augmentations = [
    ('randomly_eq', randomly_eq, {'gain_range': (-3, 3)}),  # Mild EQ changes
    ('apply_compression', apply_compression, {'threshold_db': -15, 'ratio': 3.0}),  # Moderate compression
    ('add_noise', add_noise, {'noise_type': 'pink', 'snr_db': -25})  # Pink noise at -25dB SNR
]

print("Augmentations to test:")
for name, func, args in augmentations:
    print(f"  - {name}: {args}")

Augmentations to test:
  - randomly_eq: {'gain_range': (-3, 3)}
  - apply_compression: {'threshold_db': -15, 'ratio': 3.0}
  - add_noise: {'noise_type': 'pink', 'snr_db': -25}


In [4]:
def print_audio_info(file_path, label="Audio"):
    """Print information about an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=None)
        duration = len(y) / sr
        file_size = os.path.getsize(file_path)
        print(f"{label}: {file_path}")
        print(f"  Duration: {duration:.2f}s, Sample Rate: {sr}Hz, Size: {file_size} bytes")
        return y, sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

# Test on first audio file
if audio_files:
    test_file = os.path.join(test_audio_folder, audio_files[0])
    print(f"\n=== Testing on {audio_files[0]} ===")

    # Print original file info
    y_orig, sr_orig = print_audio_info(test_file, "Original")

    # Apply each augmentation
    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{Path(audio_files[0]).stem}_{aug_name}.wav")
        print(f"\nApplying {aug_name}...")

        try:
            aug_func(test_file, output_file, aug_args)
            print_audio_info(output_file, f"Augmented ({aug_name})")
        except Exception as e:
            print(f"Error applying {aug_name}: {e}")

print("\n=== Testing complete for first file ===")


=== Testing on TRBFMXE149E3E18222.mp3 ===
Original: test_audio\TRBFMXE149E3E18222.mp3
  Duration: 202.00s, Sample Rate: 44100Hz, Size: 3232078 bytes

Applying randomly_eq...
Augmented (randomly_eq): augmentations_test_outputs\TRBFMXE149E3E18222_randomly_eq.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes

Applying apply_compression...
Augmented (apply_compression): augmentations_test_outputs\TRBFMXE149E3E18222_apply_compression.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes

Applying add_noise...
Augmented (add_noise): augmentations_test_outputs\TRBFMXE149E3E18222_add_noise.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes

=== Testing complete for first file ===


In [5]:
# Apply all augmentations to all files
print(f"\n=== Applying augmentations to all {len(audio_files)} files ===")

total_processed = 0
for audio_file in audio_files:
    input_path = os.path.join(test_audio_folder, audio_file)
    base_name = Path(audio_file).stem

    print(f"\nProcessing {audio_file}...")

    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{base_name}_{aug_name}.wav")

        try:
            aug_func(input_path, output_file, aug_args)
            total_processed += 1
            print(f"  ✓ {aug_name} -> {os.path.basename(output_file)}")
        except Exception as e:
            print(f"  ✗ {aug_name} failed: {e}")

print(f"\n=== Processing complete ===")
print(f"Total augmentations applied: {total_processed}")
print(f"Output folder: {output_folder}")
print(f"Files created: {len(os.listdir(output_folder))}")


=== Applying augmentations to all 11 files ===

Processing TRBFMXE149E3E18222.mp3...
  ✓ randomly_eq -> TRBFMXE149E3E18222_randomly_eq.wav
  ✓ apply_compression -> TRBFMXE149E3E18222_apply_compression.wav
  ✓ add_noise -> TRBFMXE149E3E18222_add_noise.wav

Processing TRBKNSX149E2E1F89F.mp3...
  ✓ randomly_eq -> TRBKNSX149E2E1F89F_randomly_eq.wav
  ✓ apply_compression -> TRBKNSX149E2E1F89F_apply_compression.wav
  ✓ add_noise -> TRBKNSX149E2E1F89F_add_noise.wav

Processing TRFVOOM127FA301CFB.mp3...
  ✓ randomly_eq -> TRFVOOM127FA301CFB_randomly_eq.wav
  ✓ apply_compression -> TRFVOOM127FA301CFB_apply_compression.wav
  ✓ add_noise -> TRFVOOM127FA301CFB_add_noise.wav

Processing TRKINMI127FA68F716.mp3...
  ✓ randomly_eq -> TRKINMI127FA68F716_randomly_eq.wav
  ✓ apply_compression -> TRKINMI127FA68F716_apply_compression.wav
  ✓ add_noise -> TRKINMI127FA68F716_add_noise.wav

Processing TRMLNPR149E35EF902.mp3...
  ✓ randomly_eq -> TRMLNPR149E35EF902_randomly_eq.wav
  ✓ apply_compression -> TRM

In [6]:
# List all output files and their information
print("=== Output Files Summary ===")
output_files = sorted(os.listdir(output_folder))
for output_file in output_files:
    file_path = os.path.join(output_folder, output_file)
    print_audio_info(file_path, output_file)

print(f"\nTotal output files: {len(output_files)}")
print("All augmentations completed successfully!")

=== Output Files Summary ===
TRBFMXE149E3E18222_add_noise.wav: augmentations_test_outputs\TRBFMXE149E3E18222_add_noise.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes
TRBFMXE149E3E18222_apply_compression.wav: augmentations_test_outputs\TRBFMXE149E3E18222_apply_compression.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes
TRBFMXE149E3E18222_randomly_eq.wav: augmentations_test_outputs\TRBFMXE149E3E18222_randomly_eq.wav
  Duration: 202.00s, Sample Rate: 22050Hz, Size: 8908460 bytes
TRBKNSX149E2E1F89F_add_noise.wav: augmentations_test_outputs\TRBKNSX149E2E1F89F_add_noise.wav
  Duration: 215.14s, Sample Rate: 22050Hz, Size: 9487916 bytes
TRBKNSX149E2E1F89F_apply_compression.wav: augmentations_test_outputs\TRBKNSX149E2E1F89F_apply_compression.wav
  Duration: 215.14s, Sample Rate: 22050Hz, Size: 9487916 bytes
TRBKNSX149E2E1F89F_randomly_eq.wav: augmentations_test_outputs\TRBKNSX149E2E1F89F_randomly_eq.wav
  Duration: 215.14s, Sample Rate: 22050Hz, Size: 948

## Training on Jams


In [ ]:
#!pip install git+https://github.com/andreamust/consonance-ACE.git

In [ ]:
#%python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#%python -m pip install jams librosa soundfile

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [2]:
import torch

def check_blackwell():
    print(f"--- Diagnóstico de Hardware ---")
    if torch.cuda.is_available():
        print(f"✅ ¡Conectado a la GPU!")
        print(f"Tarjeta: {torch.cuda.get_device_name(0)}")
        print(f"Capacidad de Cómputo: {torch.cuda.get_device_capability(0)}")
        
        # Test rápido: Multiplicación de matrices en GPU
        x = torch.randn(1000, 1000).cuda()
        y = torch.randn(1000, 1000).cuda()
        z = torch.matmul(x, y)
        print(f"🚀 Test de computación completado en la GPU.")
    else:
        print("❌ Sigue detectando CPU. Revisa que el entorno virtual esté activo.")

if __name__ == "__main__":
    check_blackwell()

--- Diagnóstico de Hardware ---
✅ ¡Conectado a la GPU!
Tarjeta: NVIDIA GeForce RTX 5060 Laptop GPU
Capacidad de Cómputo: (12, 0)
🚀 Test de computación completado en la GPU.


c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [3]:
# Login to Weights & Biases for experiment tracking
import wandb
wandb.login()
import os
#os.environ['WANDB_API_KEY'] = 'api'

c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\User\_netrc.
wandb: Currently logged in as: rafaeleduardo-moncayo01 (rafaeleduardo-moncayo01-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
JAMS_FOLDER = "jams" # Ensure you set this to the folder where your JAMS files are located
CORPORA = ["MARL-Chords", "Billboard-Chords", "Isophonics"]

df = u.load_chord_data(JAMS_FOLDER)
for each in CORPORA:
	a.print_chord_stats(u.get_chord_stats(df,[each]), title=f"{each} set statistics")

MARL-Chords set statistics
Total chords: 1,322
Unique chords: 165

Billboard-Chords set statistics
Total chords: 5,079
Unique chords: 200

Isophonics set statistics
Total chords: 890
Unique chords: 84



### PreProcess data

In [8]:
# Configuración
INPUT_FOLDER = "test_audio"
OUTPUT_FOLDER = "training_jams_data\\preprocessed_data"
SR = 22050  # Frecuencia de muestreo estándar
HOP_LENGTH = 512
BINS_PER_OCTAVE = 12
N_BINS = 84 # 7 octavas (estándar para música)

def preprocess_cqt():
    # Crear carpeta de salida si no existe
    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    
    audio_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(('.mp3', '.wav'))]
    print(f"Encontrados {len(audio_files)} archivos para procesar.")

    for file_name in audio_files:
        try:
            input_path = os.path.join(INPUT_FOLDER, file_name)
            output_path = os.path.join(OUTPUT_FOLDER, f"{Path(file_name).stem}.pt")
            
            # 1. Cargar audio
            y, sr = librosa.load(input_path, sr=SR)
            
            # 2. Calcular CQT (Magnitud)
            # Usamos np.abs para obtener la magnitud y luego pasamos a escala logarítmica (decibelios)
            cqt = librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH, 
                              n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE)
            cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
            
            # 3. Convertir a Tensor de PyTorch y mover a GPU (opcional para guardar)
            # Guardamos como float32 para ahorrar espacio pero mantener precisión
            cqt_tensor = torch.from_numpy(cqt_db).float()
            
            # 4. Guardar archivo .pt
            torch.save(cqt_tensor, output_path)
            print(f"✅ Procesado: {file_name} -> {cqt_tensor.shape}")

        except Exception as e:
            print(f"❌ Error procesando {file_name}: {e}")

if __name__ == "__main__":
    preprocess_cqt()
    print(f"\nProceso finalizado. Archivos guardados en: {OUTPUT_FOLDER}")

Encontrados 11 archivos para procesar.
✅ Procesado: TRBFMXE149E3E18222.mp3 -> torch.Size([84, 8700])
✅ Procesado: TRBKNSX149E2E1F89F.mp3 -> torch.Size([84, 9266])
✅ Procesado: TRFVOOM127FA301CFB.mp3 -> torch.Size([84, 12264])
✅ Procesado: TRKINMI127FA68F716.mp3 -> torch.Size([84, 11534])
✅ Procesado: TRMLNPR149E35EF902.mp3 -> torch.Size([84, 13717])
✅ Procesado: TRPKMIZ127F8DE2737.mp3 -> torch.Size([84, 10614])
✅ Procesado: TRQHXNX127FA620029.mp3 -> torch.Size([84, 15945])
✅ Procesado: TRQUISX127FA57F892.mp3 -> torch.Size([84, 10478])
✅ Procesado: TRUEIEJ127FA6DE0E5.mp3 -> torch.Size([84, 17050])
✅ Procesado: TRYNQFC149E3E14844.mp3 -> torch.Size([84, 13665])
✅ Procesado: TRZRAGD149E37275D3.mp3 -> torch.Size([84, 5880])

Proceso finalizado. Archivos guardados en: training_jams_data\preprocessed_data


In [ ]:
# Configuración
INPUT_FOLDER = "augmentations_test_outputs"
OUTPUT_FOLDER = "training_plus_augmentaiton\\preprocessed_data"
SR = 22050  # Frecuencia de muestreo estándar
HOP_LENGTH = 512
BINS_PER_OCTAVE = 12
N_BINS = 84 # 7 octavas (estándar para música)

def preprocess_cqt():
    # Crear carpeta de salida si no existe
    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    
    audio_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(('.mp3', '.wav'))]
    print(f"Encontrados {len(audio_files)} archivos para procesar.")

    for file_name in audio_files:
        try:
            input_path = os.path.join(INPUT_FOLDER, file_name)
            output_path = os.path.join(OUTPUT_FOLDER, f"{Path(file_name).stem}.pt")
            
            # 1. Cargar audio
            y, sr = librosa.load(input_path, sr=SR)
            
            # 2. Calcular CQT (Magnitud)
            # Usamos np.abs para obtener la magnitud y luego pasamos a escala logarítmica (decibelios)
            cqt = librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH, 
                              n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE)
            cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
            
            # 3. Convertir a Tensor de PyTorch y mover a GPU (opcional para guardar)
            # Guardamos como float32 para ahorrar espacio pero mantener precisión
            cqt_tensor = torch.from_numpy(cqt_db).float()
            
            # 4. Guardar archivo .pt
            torch.save(cqt_tensor, output_path)
            print(f"✅ Procesado: {file_name} -> {cqt_tensor.shape}")

        except Exception as e:
            print(f"❌ Error procesando {file_name}: {e}")

if __name__ == "__main__":
    preprocess_cqt()
    print(f"\nProceso finalizado. Archivos guardados en: {OUTPUT_FOLDER}")

### Training cells

In [ ]:
from ACE.trainer import main as ace_train_model

# for local
train_data_path = "training_jams_data\preprocessed_data"
vocab_path = "training_jams_data\chords_vocab.joblib"
checkpoint_path = "training_jams_data\Checkpoints"
checkpoint_decomposed_path = "training_jams_data\Decomposed_Checkpoints"
